**Попытка попробовать другую модель # 2**

Ниже мои попытки объединить метод матричной факторизации SVD с простой последовательной нейронной сетью. Такой вид модели показывает результат чуть лучше чистого SVD, но всё же хуже изначально опробованного чистого ALS. Данные для обучения те же, что в модели ALS (см.самый первый ноутбук EDA + ALS, где показано, на основе чего я их получила).  Т.к. модель работала хуже ALS, далее пришла к выводу, что лучше попробовать комбинации ALS + NN.

In [1]:
import os
import csv
import pandas as pd
import numpy as np
import matplotlib
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import SVD
from sklearn.preprocessing import MinMaxScaler
from surprise.model_selection import cross_validate, PredefinedKFold, GridSearchCV
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.models import *
from tensorflow.keras.layers import *
from itertools import product
import tensorflow as tf

2025-02-28 14:02:00.877473: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-02-28 14:02:00.877507: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-02-28 14:02:00.877978: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-28 14:02:00.881058: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-02-28 14:02:01.314216: W tensorflow/compiler/tf2

In [2]:
#Напишем базовую комплектацию показателей для проверки точности вычисления рейтинга моделью.
def Avg_Precision_at_n(fact, predicted, n=10):
    """
    Вычисление средней точности по n-позициям для двух списков значений.
    
    Вход
    ----------
    fact : list
             Фактический список элементов, который нужно предсказать.
    predicted : list
             Список, полученный в результате предсказания модели (строго упорядоченный).
    n : int
             Максимальное количество предсказываемых позиций
    Выход
    -------
    score : double
            Средняя точность на n-позициях
    """
    if not fact:
        return 0.0

    if len(predicted)>n:
        predicted = predicted[:n]

    score = 0.0
    num_hits = 0.0

    for i,p in enumerate(predicted):
        # Первое условие проверяет наличие предсказания в списке фактических элементов
        # второе условие - проверка на отсутствие (или наличие) повторов в предсказании
        if p in fact and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i+1.0)

    return score / min(len(fact), n)

def MAP_at_n(fact, predicted, n=10):
    """
    Вычисление mean average precision at n для двух списков элементов.
   
    Вход
    ----------
    fact : list
            Фактический список элементов, который нужно предсказать.
    predicted : list
            Список, полученный в результате предсказания модели (строго упорядоченный).
    n : int
            Максимальное количество предсказываемых позиций
    Выход
    -------
    score : double
            Mean average precision at n двух списков
    """
    return np.mean([Avg_Precision_at_n(f,p,n) for f,p in zip(fact, predicted)])

In [3]:
#Для подачи в модели:
files_dir = './data/recsys/'

TRAIN_CSV_PATH = os.path.join(files_dir, 'train_df.csv')
TEST_CSV_PATH = os.path.join(files_dir, 'test_df.csv')

reader = Reader(line_format="user item rating", sep = '\t',rating_scale=(1, 10)) # Зададим разброс оценок

folds_files = [(TRAIN_CSV_PATH,TEST_CSV_PATH)] #список путей к файлам для подачи в объект библиотеки surprise lib
data = Dataset.load_from_folds(folds_files, reader=reader) #создадим data-объект

pkf = PredefinedKFold() #создадим объект, позволяющий подать в модель собственный набор train и test данных
trainset, testset = next(pkf.split(data)) #определим train и test сеты

В результате мы чуть увеличили точность предсказния по сравнению с чистой версией SVD, но ALS с задачей справился всё-таки получше.

In [4]:
#Обучим SVD для получения эмбеддингов:
svd = SVD(n_epochs = 20, n_factors=1, lr_all = 0.002,reg_all = 0.08, random_state=999, verbose=False)
#Предсказание
predictions = svd.fit(trainset)

In [5]:
#Загрузим нужные нам данные:
train_df = pd.read_csv(TRAIN_CSV_PATH, sep ='\t', names = ['user_id', 'product_id','real_rating'])
test_df = pd.read_csv(TEST_CSV_PATH, sep ='\t', names = ['user_id', 'product_id','real_rating'])

In [6]:
#Объединение полученных с помощью SVD-разложения матриц в один датафрейм
df_pu = pd.DataFrame({'pu': [i for i in svd.pu], 'user_id': train_df.user_id.unique()})#Датафрейм из эмбедингов пользователей
df_qi = pd.DataFrame({'qi': [i for i in svd.qi], 'product_id':train_df.product_id.unique()})#Датафрейм из эмбедингов обьектов

#Поменяем тип данных в исходном df со строкового на целочислденный
train_df['user_id'] =  train_df.user_id.astype(int)
train_df['product_id'] =  train_df.product_id.astype(int)
# #Объединим результаты:
train_df = train_df.merge(df_pu, on = 'user_id')#Обеденили основной датафрейм с пользовательскими эмбедингами
train_df = train_df.merge(df_qi, on = 'product_id')#И с эмбедингами обьектов
train_df.head()#посмотрим результат

,user_id,product_id,real_rating,pu,qi
0,1,196,10.0000,[0.02359618213155825],[0.2134981144051755]
1,1,10258,7.9000,[0.02359618213155825],[-0.11367433863934083]
2,1,10326,1.1875,[0.02359618213155825],[0.01723478305726886]
3,1,12427,8.8700,[0.02359618213155825],[0.10053183709522633]
4,1,13032,1.7490,[0.02359618213155825],[-0.10227479865565285]


In [7]:
#Определим столбцы X и Y
train_df['X'] = train_df.apply(lambda x: np.concatenate((x['pu'], x['qi'])), axis=1)#Обьеденяем эмбединги
train_df['y'] = train_df['real_rating']
#И сами X и Y
X_train, y_train = train_df['X'].values, train_df['y'].values
X_train = [list(i) for i in X_train]
X_train = np.array(X_train)
y_train = np.array(y_train)

In [8]:
#Добавим эмбединг в тестовый сет, чтобы на его основе потом сделать итоговое предсказание:
test_df = test_df.merge(train_df[['X','user_id','product_id']], how = "right")

In [9]:
#Определим X_test
X_test = test_df['X'].values
X_test = [list(i) for i in X_test]

In [10]:
#Создадим чекпойнт:
cpt_path ='checkpoints/instacart/svd_checkpoint_rec.h5'
checkpoint = ModelCheckpoint(cpt_path, monitor='loss', verbose=1, save_best_only=True, mode='min', 
                                                save_freq = 'epoch')
stopper = EarlyStopping(monitor = 'loss', verbose = 1, restore_best_weights=True, mode="min", patience = 15)

In [11]:
def recommender_model(min_rating = 1, max_rating = 10, learning_rate=1e-3):
    input_embedding = Input(shape=(2,))
    x = Dense(256, activation='relu')(input_embedding)
    x = Dropout(0.2)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.2)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.2)(x) #и еще один dropout
    x = Dense(1,activation = 'sigmoid', kernel_initializer='lecun_uniform')(x)
    out = Lambda(lambda x: x * (max_rating - min_rating) + min_rating)(x)
    
    model = Model(inputs=input_embedding, outputs=out)
    
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    loss=tf.keras.losses.MSE

    model.compile(loss=loss, optimizer=optimizer, metrics = ['accuracy'])
    return model

In [12]:
rec_model = recommender_model()
rec_model.summary()

2025-02-28 10:19:22.926665: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-02-28 10:19:22.959761: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-02-28 10:19:22.959799: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-02-28 10:19:22.961555: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-02-28 10:19:22.961590: I external/local_xla/xla/stream_executor

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 2)]               0         
                                                                 
 dense (Dense)               (None, 256)               768       
                                                                 
 dropout (Dropout)           (None, 256)               0         
                                                                 
 dense_1 (Dense)             (None, 128)               32896     
                                                                 
 dropout_1 (Dropout)         (None, 128)               0         
                                                                 
 dense_2 (Dense)             (None, 64)                8256      
                                                                 
 dropout_2 (Dropout)         (None, 64)                0     

In [13]:
#Обучим её:
history = rec_model.fit(X_train, y_train, epochs=100, batch_size=64, callbacks=[checkpoint, stopper])

Epoch 1/100


2025-02-28 10:19:24.476834: I external/local_xla/xla/service/service.cc:168] XLA service 0x7f7b3916af80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-02-28 10:19:24.476858: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 3050, Compute Capability 8.6
2025-02-28 10:19:24.487932: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-02-28 10:19:24.516608: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907
I0000 00:00:1740727164.572800   18793 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


106941/106941 [==============================] - ETA: 0s - loss: 2.8376 - accuracy: 0.0149
Epoch 1: loss improved from inf to 2.83764, saving model to checkpoints/instacart/svd_checkpoint_rec.h5
106941/106941 [==============================] - 424s 4ms/step - loss: 2.8376 - accuracy: 0.0149
Epoch 2/100
    39/106941 [..............................] - ETA: 7:17 - loss: 2.6843 - accuracy: 0.0164

/home/nette/miniconda3/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


106935/106941 [============================>.] - ETA: 0s - loss: 2.8288 - accuracy: 0.0149
Epoch 2: loss improved from 2.83764 to 2.82882, saving model to checkpoints/instacart/svd_checkpoint_rec.h5
106941/106941 [==============================] - 427s 4ms/step - loss: 2.8288 - accuracy: 0.0149
Epoch 3/100
106934/106941 [============================>.] - ETA: 0s - loss: 2.8289 - accuracy: 0.0149
Epoch 3: loss did not improve from 2.82882
106941/106941 [==============================] - 421s 4ms/step - loss: 2.8289 - accuracy: 0.0149
Epoch 4/100
106939/106941 [============================>.] - ETA: 0s - loss: 2.8282 - accuracy: 0.0149
Epoch 4: loss improved from 2.82882 to 2.82815, saving model to checkpoints/instacart/svd_checkpoint_rec.h5
106941/106941 [==============================] - 427s 4ms/step - loss: 2.8282 - accuracy: 0.0149
Epoch 5/100
106929/106941 [============================>.] - ETA: 0s - loss: 2.8286 - accuracy: 0.0149
Epoch 5: loss did not improve from 2.82815
106941/

In [10]:
#Загрузим модель с наилучшим результатом:
cpt_path ='checkpoints/instacart/svd_checkpoint_rec.h5'
rec_model = tf.keras.models.load_model(cpt_path)

2025-02-28 14:03:51.961192: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-02-28 14:03:52.129729: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-02-28 14:03:52.129768: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-02-28 14:03:52.132688: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-02-28 14:03:52.132738: I external/local_xla/xla/stream_executor

In [11]:
rec_preds = rec_model.predict(X_test)

213881/213881 [==============================] - 253s 1ms/step


In [12]:
test_df['predicted'] = rec_preds
test_df.head(15)

,user_id,product_id,real_rating,X,predicted
0,1,196,10.0,"[0.02359618213155825, 0.2134981144051755]",2.732126
1,1,10258,5.0,"[0.02359618213155825, -0.11367433863934083]",2.160395
2,1,10326,NaN,"[0.02359618213155825, 0.01723478305726886]",1.992448
3,1,12427,2.0,"[0.02359618213155825, 0.10053183709522633]",2.227384
4,1,13032,3.0,"[0.02359618213155825, -0.10227479865565285]",2.124820
5,1,13176,NaN,"[0.02359618213155825, 0.9964180687198564]",4.313373
6,1,14084,NaN,"[0.02359618213155825, -0.4743862329811647]",3.081813
7,1,17122,NaN,"[0.02359618213155825, -0.0795265160824597]",2.050910
8,1,25133,6.0,"[0.02359618213155825, -0.05989147468674692]",2.002298
9,1,26088,NaN,"[0.02359618213155825, -0.06398714541346828]",2.011598


In [13]:
#Теперь нам нужно правильно отсортировать предсказание по столбцу с предполагаемым рейтингом, обрезав его до 10 значений:
test_trim = test_df[['user_id','product_id','predicted']]
test_grouped = test_trim.merge(test_trim
        #Сгруппируем по пользователям, найдем 10 макс значений предсказанного рейтинга
        .groupby('user_id').predicted.nlargest(10)
        #Получим новый df с  MultiIndex, котрый сбросим через reset_index
        .reset_index('user_id'),
        # в новом df нет колонки "product_id" поэтому необходимо объединить изначальный сет с полученным
    how='right') # при этом все строки, которых нет в новом df удалятся

#Проверка
test_grouped.head(15)

,user_id,product_id,predicted
0,1,13176,4.313373
1,1,49235,3.123010
2,1,14084,3.081813
3,1,196,2.732126
4,1,26405,2.538788
5,1,46149,2.519204
6,1,12427,2.227384
7,1,10258,2.160395
8,1,13032,2.124820
9,1,17122,2.050910


In [14]:
#Соберем все id продуктов в один столбец - pred_order:
test_result = test_grouped.groupby('user_id')['product_id'].unique().reset_index()
test_result.columns=['user_id','pred_order']
#Check
test_result.head(15)

,user_id,pred_order
0,1,"[13176, 49235, 14084, 196, 26405, 46149, 12427..."
1,2,"[24852, 21709, 27966, 13351, 13742, 37646, 482..."
2,3,"[28373, 21137, 24010, 48523, 22035, 47766, 938..."
3,7,"[13249, 13176, 42803, 49235, 45628, 26346, 115..."
4,13,"[1689, 4210, 27435, 33735, 27086, 43086, 19474..."
5,14,"[29509, 3384, 17556, 19585, 45519, 4489, 8744,..."
6,15,"[14715, 196, 30292, 11266, 12427, 27839, 37710..."
7,17,"[47031, 22309, 14146, 44492, 11919, 7026, 1853..."
8,21,"[24852, 43154, 35221, 44632, 38444, 10957, 196..."
9,22,"[27845, 13176, 35221, 36724, 32655, 5212, 4217..."


In [15]:
#Таким же образом соберем факт:
fact_df = test_df[['user_id','product_id','real_rating']]
fact_df = fact_df.merge(fact_df
        #Сгруппируем по пользователям, найдем 10 макс значений предсказанного рейтинга
        .groupby('user_id').real_rating.nlargest(10)
        #Получим новый df с  MultiIndex, котрый сбросим через reset_index
        .reset_index('user_id'),
        # в новом df нет колонки "product_id" поэтому необходимо объединить изначальный сет с полученным
    how='right') # при этом все строки, которых нет в новом df удалятся
fact_df.drop_duplicates(inplace = True)
fact_df = fact_df.groupby('user_id')['product_id'].unique().reset_index()
fact_df.columns=['user_id','fact_order']
#Check
fact_df.head(15)

,user_id,fact_order
0,1,"[196, 46149, 25133, 10258, 13032, 12427, 10326..."
1,2,"[24852, 16589, 1559, 19156, 18523, 33754, 2170..."
2,3,"[39190, 47766, 21903, 43961, 17668, 248, 1005,..."
3,7,"[47272, 29993, 31683, 27690, 13198, 30391, 376..."
4,13,"[27435, 27086, 4210, 43086, 42248, 1689, 5025,..."
5,14,"[29509, 23803, 15869, 8744, 37266, 11131, 239,..."
6,15,"[196, 48142, 1747, 10441, 11266, 12427, 14715,..."
7,17,"[7350, 18534, 9006, 26767, 14146, 16797, 812, ..."
8,21,"[18523, 31387, 28204, 27548, 196, 3023, 3485, ..."
9,22,"[35221, 24964, 7948, 24506, 2452, 4217, 4421, ..."


In [19]:
#Посмотрим на результат:
result = fact_df.merge(test_result, how = 'left')
result['fact_order'] = result['fact_order'].astype(str)
result['pred_order'] = result['pred_order'].astype(str) 
#Применим функцию MAP_at_n построчно:
result['MAP'] = result.apply(lambda x: MAP_at_n(x.fact_order, x.pred_order), axis=1)
result.head()

,user_id,fact_order,pred_order,MAP
0,1,[ 196 46149 25133 10258 13032 12427 10326 131...,[13176 49235 14084 196 26405 46149 12427 102...,0.377049
1,2,[24852 16589 1559 19156 18523 33754 21709 472...,[24852 21709 27966 13351 13742 37646 48210 321...,0.377049
2,3,[39190 47766 21903 43961 17668 248 1005 18...,[28373 21137 24010 48523 22035 47766 9387 391...,0.213115
3,7,[47272 29993 31683 27690 13198 30391 37602 211...,[13249 13176 42803 49235 45628 26346 11520 391...,0.262295
4,13,[27435 27086 4210 43086 42248 1689 5025 56...,[ 1689 4210 27435 33735 27086 43086 19474 373...,0.213115


In [20]:
sum(result.MAP)/len(result.user_id.unique())

0.29263116983581233

Предсказание стало хуже.